# Create round_info.csv

Builds `round_info.csv` from the HAL configs (notebook 01) and the FOV/boundary layout (notebook 02) -- the per-round series/HAL-config/data-dir table that notebook 04 turns into the Dave recipe.

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/before_imaging/<variant>/<acquisition>/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.acquisition.dave      import create_round_info, create_round_info_multitissue
from MERci.acquisition.positions import (
    discover_boundary_files, resolve_boundaries_source_dir, group_boundaries_by_path_mode,
)

In [2]:
SETTINGS_DIR  = SAMPLE_DIR / "settings"
METADATA_DIR  = SAMPLE_DIR / "metadata"
POSITIONS_DIR = SAMPLE_DIR / "positions"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

# SAMPLE_NAME is the TRUE top-level experiment id (e.g. "LT058_sample_07"),
# auto-detected from the folder structure -- NOT SAMPLE_DIR.name, which is only
# this acquisition's own local folder name (e.g. "merfish") once split into
# sibling acquisition-type subfolders. Must match what notebook 02 used, since
# create_round_info_multitissue below references positions_{SAMPLE_NAME}_*.txt.
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
# POSITIONS_TAG is what every positions_{...}.txt filename below/passed downstream
# uses -- SAMPLE_NAME alone in the flat layout, or SAMPLE_NAME_IMAGING_DIR split
# layout, so a sibling acquisition never collides.
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SAMPLE_NAME  : {SAMPLE_NAME}")

SAMPLE_DIR   : C:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time
SAMPLE_NAME  : 251225_LT027_saving_time


In [3]:
# ── Experiment parameters ──────────────────────────────────────
MICROSCOPE  = "ST2"   # microscope identifier

# Per-tissue path mode -- MUST match what notebook 02 actually used (same
# convention as BOUNDARY_SOURCE below): a tissue's own consecutive
# boundaries merge into one "legacy" segment (no transit) or stay separate,
# bridged by transit, under "transit". Default: "legacy" for every tissue,
# matching notebook 02's own default. Override per tissue via
# TISSUE_PATH_MODE_OVERRIDES, e.g. {2: "transit"} -- keep this in sync with
# whatever notebook 02 was actually run with, or this notebook will
# reference positions files that don't exist.
TISSUE_PATH_MODE           = "legacy"
TISSUE_PATH_MODE_OVERRIDES = {}

def tissue_path_mode(t):
    mode = TISSUE_PATH_MODE_OVERRIDES.get(t, TISSUE_PATH_MODE)
    if mode not in ("legacy", "transit"):
        raise ValueError(f"Tissue {t}: path mode must be 'legacy' or 'transit', got {mode!r}")
    return mode

# ── Detect the tissue/boundary layout (written by notebook 02) ──────
# Same source resolution as notebook 02 (positions/boundaries/{manual,from_mosaic}/)
# -- BOUNDARY_SOURCE = None auto-picks whichever has files, so this notebook
# agrees with notebook 02 without needing state passed between them. Set
# explicitly if notebook 02 was overridden manually.
BOUNDARY_SOURCE = None
BOUNDARY_DIR, BOUNDARY_SOURCE = resolve_boundaries_source_dir(POSITIONS_DIR, BOUNDARY_SOURCE)
print(f"BOUNDARY_SOURCE: {BOUNDARY_SOURCE}")
print(f"BOUNDARY_DIR   : {BOUNDARY_DIR}")

# Group boundaries the same way notebook 02 did (group_boundaries_by_path_mode)
# -- more than one resulting group -> per-segment recipe (boundary + transit
# movies); exactly one -> the classic single-positions recipe.
boundaries, MODE = discover_boundary_files(BOUNDARY_DIR)
groups           = group_boundaries_by_path_mode(boundaries, MODE, tissue_path_mode)
MULTI_BOUNDARY   = len(groups) > 1
print(f"Layout mode: {MODE}  ({len(boundaries)} boundary file(s) -> {len(groups)} segment(s)) -> "
      f"{'per-segment' if MULTI_BOUNDARY else 'single-positions'} recipe")

# HAL config filenames (from notebook 01 / SETTINGS_DIR)
# Adjust these to match the actual files created by notebook 01
bits_hal_configs    = sorted(SETTINGS_DIR.glob("hal-config-*bits*.xml"))
cells_hal_configs   = sorted(SETTINGS_DIR.glob("hal-config-*cells*.xml"))
transit_hal_configs = sorted(SETTINGS_DIR.glob("hal-config-*transit*.xml"))

print("\nAvailable HAL configs in settings/:")
for p in sorted(SETTINGS_DIR.glob("hal-config-*.xml")):
    print(f"  {p.name}")

# Set these manually if auto-detection picks the wrong files
BITS_HAL_CONFIG    = bits_hal_configs[0].name    if bits_hal_configs    else "hal-config-mf3-bits.xml"
CELLS_HAL_CONFIG   = cells_hal_configs[0].name   if cells_hal_configs   else "hal-config-mf3-cells.xml"
TRANSIT_HAL_CONFIG = transit_hal_configs[0].name if transit_hal_configs else None

print(f"\nBits    HAL config : {BITS_HAL_CONFIG}")
print(f"Cells   HAL config : {CELLS_HAL_CONFIG}")
print(f"Transit HAL config : {TRANSIT_HAL_CONFIG}")

if MULTI_BOUNDARY and TRANSIT_HAL_CONFIG is None:
    raise FileNotFoundError(
        "Multiple boundaries detected but no hal-config-*transit*.xml in settings/. "
        "Run the transit cell in notebook 01 first."
    )

BOUNDARY_SOURCE: from_mosaic
BOUNDARY_DIR   : C:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\positions\boundaries\from_mosaic
Layout mode: single  (4 boundary file(s) -> 1 segment(s)) -> single-positions recipe

Available HAL configs in settings/:
  hal-config-st2-10x-mosaic-405.xml
  hal-config-st2-60x-mosaic-405.xml
  hal-config-st2-bits-blkf15_488f2_560f141_650f141.xml
  hal-config-st2-bits-blkf21_488f203_560f201_650f201.xml
  hal-config-st2-cells-blkf11_405f201_488f203.xml
  hal-config-st2-cells-blkf15_405f141_488f143.xml
  hal-config-st2-focustest-blkf2_488f1.xml
  hal-config-st2-transit-blkf2.xml

Bits    HAL config : hal-config-st2-bits-blkf15_488f2_560f141_650f141.xml
Cells   HAL config : hal-config-st2-cells-blkf11_405f201_488f203.xml
Transit HAL config : hal-config-st2-transit-blkf2.xml


## Round – bit – color mapping

Define the round → bit → colour mapping for the codebook. This is the single
source of **`N_HYBS`** (the number of hybridisation/bits rounds, taken as the max
round index) used by the recipe below, and it is saved to `round_bit_color_map.csv`
for notebook 05 to reuse (data organization + Dave annotation).

In [4]:
# round : hyb/bit index (1-indexed), matching the bits movie series number
#         (hal-{mic}_01, _02, …); NOT the Dave imaging-round number.
# bit   : bit number     |     color : excitation wavelength (nm)
round_bit_color = [
    (1,  1,  647), (1,  2,  560),
    (2,  3,  560), (2,  4,  647),
    (3,  5,  647), (3,  6,  560),
    (4,  7,  647), (4,  8,  560),
    (5,  9,  560), (5,  10, 647),
    (6,  11, 647), (6,  12, 560),
    (7,  13, 647), (7,  14, 560),
    (8,  15, 560), (8,  16, 647),
    (9,  17, 647), (9,  18, 560),
    (10, 19, 647), (10, 20, 560),
    (11, 21, 560), (11, 22, 647),
    (12, 23, 647), (12, 24, 560),
    (13, 25, 647), (13, 26, 560),
]

rbc_df   = pd.DataFrame(round_bit_color, columns=["round", "bit", "color"])
rbc_path = METADATA_DIR / "round_bit_color_map.csv"
rbc_df.to_csv(rbc_path, index=False)

N_HYBS = int(rbc_df["round"].max())   # number of bits rounds, derived from the mapping
print(f"Saved: {rbc_path}")
print(f"N_HYBS (from mapping): {N_HYBS}")
print(rbc_df.to_string(index=False))

Saved: C:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\metadata\round_bit_color_map.csv
N_HYBS (from mapping): 13
 round  bit  color
     1    1    647
     1    2    560
     2    3    560
     2    4    647
     3    5    647
     3    6    560
     4    7    647
     4    8    560
     5    9    560
     5   10    647
     6   11    647
     6   12    560
     7   13    647
     7   14    560
     8   15    560
     8   16    647
     9   17    647
     9   18    560
    10   19    647
    10   20    560
    11   21    560
    11   22    647
    12   23    647
    12   24    560
    13   25    647
    13   26    560


In [5]:
if MULTI_BOUNDARY:
    round_info = create_round_info_multitissue(
        microscope         = MICROSCOPE,
        n_bits             = N_HYBS,
        bits_hal_config    = BITS_HAL_CONFIG,
        cells_hal_config   = CELLS_HAL_CONFIG,
        transit_hal_config = TRANSIT_HAL_CONFIG,
        sample_dir         = SAMPLE_DIR,
        boundaries         = boundaries,
        mode               = MODE,
        sample_name        = POSITIONS_TAG,
        tissue_path_mode   = tissue_path_mode,
    )
else:
    round_info = create_round_info(
        microscope       = MICROSCOPE,
        n_bits           = N_HYBS,
        bits_hal_config  = BITS_HAL_CONFIG,
        cells_hal_config = CELLS_HAL_CONFIG,
        sample_dir       = SAMPLE_DIR,
        positions_txt    = POSITIONS_DIR / f"positions_{POSITIONS_TAG}.txt",
    )

print(round_info.to_string(index=False))

out_csv = METADATA_DIR / "round_info.csv"
round_info.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")

 imaging_round imaging_type                  series                                           hal_config                                                                                          data_dir
             1        cells hal-st2-cells_{fov:04d}      hal-config-st2-cells-blkf11_405f201_488f203.xml    C:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\data\cells
             2         bits    hal-st2_01_{fov:04d} hal-config-st2-bits-blkf15_488f2_560f141_650f141.xml C:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\data\hybs\H01
             3         bits    hal-st2_02_{fov:04d} hal-config-st2-bits-blkf15_488f2_560f141_650f141.xml C:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\data\hybs\H02
             4         bits    hal-st2_03_{fov:04d} hal-config-st2-bits-blkf15_488f2_560f141_650f141.xml C:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time